## Diffusion Transformer Flow Model — MNIST

Trains a DiT-based flow-matching model to generate MNIST digits, then samples from it with classifier-free guidance.

In [ ]:
import os
import random
import numpy as np
import torch
from matplotlib import pyplot as plt

from data_sampler import MNISTSampler
from diffusion_transformer_flow_model import DiffusionTransformerFlowModel
from simulate_output import visualize_output, checkpoint

### 1. Flow Matching Objective

Defines the noising schedule, target vector field, and training-step loss.

In [ ]:
def alpha(t: torch.Tensor) -> torch.Tensor:
    return t

def alpha_dot(t: torch.Tensor) -> torch.Tensor:
    return torch.ones_like(t)
    
def beta(t: torch.Tensor) -> torch.Tensor:
    return 1 - t

def beta_dot(t: torch.Tensor) -> torch.Tensor:
    return -torch.ones_like(t)

In [ ]:
def conditional_vector_field(x, z, t):
    b = t.shape[0]

    # Calculate alpha_t, beta_t, and their derivatives
    alpha_t = alpha(t).reshape(b, 1, 1, 1)
    beta_t = beta(t).reshape(b, 1, 1, 1)
    alpha_dot_t = alpha_dot(t).reshape(b, 1, 1, 1)
    beta_dot_t = beta_dot(t).reshape(b, 1, 1, 1)

    return (alpha_dot_t - beta_dot_t/beta_t * alpha_t)*z + beta_dot_t/beta_t * x

In [ ]:
def set_x(z, t):
    b = t.shape[0]

    # Sample noise
    epsilon = torch.randn_like(z)

    # Calculate alpha_t and beta_t
    alpha_t = alpha(t).reshape(b, 1, 1, 1)
    beta_t = beta(t).reshape(b, 1, 1, 1)

    return alpha_t * z + beta_t * epsilon

![](../figures/algorithm5.png)

In [ ]:
def training_step(model, optimizer, data_sampler, batch_size, p = 0.35, 
                  eps = 0.001):
    # Sample a data example (z, y)
    z, y = data_sampler.sample(batch_size) # Not sure if this implementation is the most efficient
    
    # Sample time, avoiding 1 as this can cause numberical issues
    t = torch.rand(batch_size).to(z) * (1 - eps)

    # Sample noise and set x
    x = set_x(z, t)
    
    # Set each label to 10 (our null) with probability p
    xi = torch.rand(y.shape[0]).to(y.device)
    y[xi < p] = 10

    # Compute loss
    ut_theta = model(x, t, y)
    ut_target = conditional_vector_field(x, z, t)
    loss = torch.square(ut_theta - ut_target).mean()

    # Update the model parameters
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    
    return float(loss.detach().item())

### 2. Train the Model

Start here to actually run something: sets up the device, seed, hyperparameters, model, and optimizer, then runs the training loop.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
n_steps = 1000 #20000
warmup_steps = 200 #500
lr = 0.4e-3
batch_size = 256
ckpt_every = 1000
data_dir = "../data"
output_dir = "../output"

os.makedirs(output_dir, exist_ok=True)

In [ ]:
data_sampler = MNISTSampler(data_dir).to(device)

In [ ]:
model = DiffusionTransformerFlowModel(
    img_size = 32,
    patch_size = 4,
    num_layers = 8,
    c = 1,
    dim = 256,
    heads = 8,
    final_dim = 10,
    n_classes = 11,
).to(device)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

Start here - training loop

In [ ]:
model.train()

for pg in optimizer.param_groups:
    pg["lr"] = 0.0

losses = []

for step in range(n_steps):
    # Linearly ramp lr from 0 to target over warmup_steps, then hold constant.
    # Warmup prevents large destructive updates early in training when gradients are noisy
    # and the optimizer momentum/variance estimates (m, v in Adam) have not yet stabilized.
    if warmup_steps > 0 and step < warmup_steps:
        cur_lr = lr * float(step + 1) / float(warmup_steps)
    else:
        cur_lr = lr
    for pg in optimizer.param_groups:
        pg["lr"] = cur_lr

    loss = training_step(model, optimizer, data_sampler, batch_size, p = 0.35, 
                         eps = 0.001)
    losses.append(loss)

    if (step + 1) % 100 == 0:
        print(f"Step {step+1}/{n_steps}")

    # Callback if specified
    if ckpt_every is not None and step % ckpt_every == 0 and step > 0:
        model.eval()
        checkpoint(model, optimizer, step, output_dir, device)
        model.train()

### 3. Results

Plots the training loss and samples digits from the trained model at a few guidance scales.

In [ ]:
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Loss")
plt.show()

In [ ]:
samples_per_class = 10
num_timesteps = 100
guidance_scales = [1.0, 3.0, 5.0]

model.eval()

visualize_output(
    model=model,
    device=device,
    samples_per_class=samples_per_class,
    num_timesteps=num_timesteps,
    guidance_scales=guidance_scales,
)
plt.show()